# Tin tức (Unstructured Data) — One-layer Ingestion

Notebook điều khiển luồng ingestion 1 tầng cho 2 nguồn `rss`, `html`.

Mỗi nguồn ghi output riêng theo layout:
- `news/<source>/date=<run_date>/PART-000.parquet`

Partition theo ngày (date=YYYY-MM-DD). Khi rerun cùng ngày có thể truncate partition rồi ghi file mới.

Schema của output thống nhất theo `NEWS_COLUMNS`, phục vụ downstream sentiment analysis.

In [1]:
import os, sys, warnings, threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

_orig_hook = threading.excepthook
def _quiet_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass
    else:
        _orig_hook(args)
threading.excepthook = _quiet_hook
warnings.filterwarnings("ignore")

from pathlib import Path
root = Path.cwd()
if not (root / "ingestion").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from ingestion.common import (
    configure_logging,
    load_dotenv_from_project_root,
 )
from ingestion.unstructured_data import (
    NEWS_COLUMNS,
    NewsIngestionConfig,
    ingest_news,
    validate_news_df,
 )

configure_logging()
load_dotenv_from_project_root()
print("[OK] setup (RSS+HTML only mode)")

2026-04-18 13:27:00 [INFO] Đã nạp biến môi trường từ C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\.env


[OK] setup (RSS+HTML only mode)


## Cấu hình

In [2]:
rate_limit = int(os.getenv("NEWS_RATE_LIMIT_RPM", "60"))

news_cfg = NewsIngestionConfig(
    use_listing_tickers=True,
    listing_exchange_filter=["HSX", "HNX", "UPCOM"],
    max_tickers_per_run=50,
    max_articles_per_source=200,
    rss_max_per_feed=200,
    html_max_per_source=200,
    days_back=7,
    days_back_rss=7,
    days_back_html=7,
    strict_published_at_days_back=False,
    rate_limit_rpm=rate_limit,
    enable_rss=True,
    enable_html=True,
    enable_ticker_match=True,
    append_only=False,
    truncate_partition=True,
)

print(f"run_date     : {news_cfg.run_date}")
print(f"news_root    : {news_cfg.news_root}")
print(f"sources.yaml : {news_cfg.resolved_sources_yaml()}")
print(f"listing      : {news_cfg.resolved_listing_parquet()}")
print(f"listing_ok   : {news_cfg.resolved_listing_parquet().is_file()}")
print(f"tickers      : {len(news_cfg.resolved_tickers())}")
print(f"days_back    : rss={news_cfg.days_back_rss}d html={news_cfg.days_back_html}d (fallback={news_cfg.days_back}, strict={news_cfg.strict_published_at_days_back})")
print(f"max_per      : rss={news_cfg.rss_max_per_feed} html={news_cfg.html_max_per_source}")
print(f"focus_mode   : RSS+HTML only (rss_enabled={news_cfg.enable_rss} html_enabled={news_cfg.enable_html})")
print(f"rate_limit   : {news_cfg.rate_limit_rpm} rpm")
print(f"write_mode   : append_only={news_cfg.append_only} (truncate_partition={news_cfg.truncate_partition})")
print(f"schema       : {NEWS_COLUMNS}")

run_date     : 2026-04-18
news_root    : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news
sources.yaml : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\ingestion\unstructured_data\sources.yaml
listing      : C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet
listing_ok   : True
tickers      : 50
days_back    : rss=7d html=7d (fallback=7, strict=False)
max_per      : rss=200 html=200
focus_mode   : RSS+HTML only (rss_enabled=True html_enabled=True)
rate_limit   : 60 rpm
write_mode   : append_only=False (truncate_partition=True)
schema       : ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


## Chạy

In [3]:
news_paths = ingest_news(news_cfg)
news_paths

2026-04-18 13:27:00 [INFO] ingest_news mode: RSS+HTML only
2026-04-18 13:27:02 [INFO] RSS feed=https://vnexpress.net/rss/kinh-doanh.rss entries_found=60 rows_added=60 take<=200
2026-04-18 13:27:02 [INFO] RSS feed=https://vnexpress.net/rss/kinh-doanh/chung-khoan.rss entries_found=0 rows_added=0 take<=200
2026-04-18 13:27:04 [INFO] RSS feed=https://cafef.vn/home.rss entries_found=60 rows_added=60 take<=200
2026-04-18 13:27:05 [INFO] RSS feed=https://cafef.vn/xa-hoi.rss entries_found=50 rows_added=50 take<=200
2026-04-18 13:27:06 [INFO] RSS feed=https://cafef.vn/bat-dong-san.rss entries_found=50 rows_added=50 take<=200
2026-04-18 13:27:07 [INFO] RSS feed=https://cafef.vn/doanh-nghiep.rss entries_found=50 rows_added=50 take<=200
2026-04-18 13:27:08 [INFO] RSS feed=https://cafef.vn/tai-chinh-ngan-hang.rss entries_found=50 rows_added=50 take<=200
2026-04-18 13:27:09 [INFO] RSS feed=https://cafef.vn/tai-chinh-quoc-te.rss entries_found=50 rows_added=50 take<=200
2026-04-18 13:27:10 [INFO] RSS 

{'row_counts': {'rss': 634, 'html': 99},
 'rss': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\rss\\date=2026-04-18\\PART-000.parquet'},
 'html': {'parquet': 'C:\\Users\\ACER\\Downloads\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Unstructure_Data\\news\\html\\date=2026-04-18\\PART-000.parquet'}}

## Kiểm tra kết quả

In [4]:
import pandas as pd

print("=" * 70)
print("OUTPUT FILES")
print("=" * 70)
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        print(f"  {key:8s}: (trống)")
        continue
    df = pd.read_parquet(parquet_path)
    print(f"  {key:8s}: {len(df):>5d} dòng")
    print(f"    parquet: {parquet_path}")

print(f"\nRow counts: {news_paths.get('row_counts', {})}")

OUTPUT FILES
  rss     :   634 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\rss\date=2026-04-18\PART-000.parquet
  html    :    99 dòng
    parquet: C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\Unstructure_Data\news\html\date=2026-04-18\PART-000.parquet

Row counts: {'rss': 634, 'html': 99}


## Validation & Data Quality

In [5]:
# Validate per-source output with NEWS_COLUMNS
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        print(f"{key}: no output")
        continue

    df = pd.read_parquet(parquet_path)
    issues = validate_news_df(df)
    print(f"{key}: {len(df)} rows")
    if issues:
        print("  ⚠ issues:")
        for i in issues:
            print(f"    - {i}")
    else:
        print("  ✓ validation passed")
    print(f"  columns: {list(df.columns)}")

rss: 634 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']
html: 99 rows
  ✓ validation passed
  columns: ['article_id', 'source', 'ticker', 'title', 'summary', 'body_text', 'url', 'published_at', 'fetched_at', 'language', 'raw_ref']


In [6]:
# Quick preview for sentiment-ready fields
for key in ["rss", "html"]:
    info = news_paths.get(key, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        continue
    df = pd.read_parquet(parquet_path)
    print(f"\n[{key}] preview")
    print(df[["article_id", "source", "ticker", "title", "body_text", "published_at"]].head(3))


[rss] preview
                                          article_id  \
0  48032fd3f60634d49cbcaed238b5ae21f5f4e34feae938...   
1  40bf320b319826dadde4691c57fa8962d3df10c19381eb...   
2  85fa78bf47114ddd777935130b0fe24bb792c2c68bc00d...   

                     source ticker  \
0  vnexpress_kinh_doanh_rss    NaN   
1  vnexpress_kinh_doanh_rss    NaN   
2  vnexpress_kinh_doanh_rss    NaN   

                                               title body_text  \
0                   Giá vàng miếng tăng 1 triệu đồng             
1  Thập kỷ tiến nhanh của hợp tác kinh tế Việt Na...             
2  Thách thức nhiên liệu vừa đắt vừa thiếu của hà...             

               published_at  
0 2026-04-18 03:36:05+00:00  
1 2026-04-17 22:00:00+00:00  
2 2026-04-17 17:36:04+00:00  

[html] preview
                                          article_id          source ticker  \
0  86fd7ed9fca3d733a6cf8ae02a40cb08e222ee2a497531...  vnexpress_html    NaN   
1  776b0edfe50e764c0ba5ef222eac1ac6273da5b3856ea

In [7]:
# Quality report: row count, nulls, published_at range for RSS + HTML
import pandas as pd

sources = ["rss", "html"]
main_cols = ["article_id", "title", "url", "published_at"]

report_rows = []
for src in sources:
    info = news_paths.get(src, {})
    parquet_path = info.get("parquet", "")
    if not parquet_path:
        report_rows.append(
            {
                "source": src,
                "rows": 0,
                "min_published_at": None,
                "max_published_at": None,
                **{f"null_{c}": None for c in main_cols},
            }
        )
        continue

    df = pd.read_parquet(parquet_path)
    row = {"source": src, "rows": int(len(df))}
    for col in main_cols:
        row[f"null_{col}"] = int(df[col].isna().sum()) if col in df.columns else int(len(df))

    if "published_at" in df.columns:
        pub = pd.to_datetime(df["published_at"], errors="coerce", utc=True)
        if pub.notna().any():
            row["min_published_at"] = str(pub.min())
            row["max_published_at"] = str(pub.max())
        else:
            row["min_published_at"] = None
            row["max_published_at"] = None
    else:
        row["min_published_at"] = None
        row["max_published_at"] = None

    report_rows.append(row)

quality_df = pd.DataFrame(report_rows)
print("QUALITY REPORT")
display(quality_df)

counts = {r["source"]: int(r["rows"]) for r in report_rows}
if counts.get("rss", 0) == 0 or counts.get("html", 0) == 0:
    print("⚠ Warning: RSS hoặc HTML đang rỗng — cần kiểm tra lại feed/selector.")
else:
    print("✓ RSS + HTML balance looks acceptable for this run.")

QUALITY REPORT


,source,rows,null_article_id,null_title,null_url,null_published_at,min_published_at,max_published_at
0,rss,634,0,0,0,0,2026-04-11 06:30:00+00:00,2026-04-18 06:23:00+00:00
1,html,99,0,0,0,99,NaN,NaN


✓ RSS + HTML balance looks acceptable for this run.
